# Notebook 05: Extensions 1 & 2 — Sensitivity Analysis & Seasonality Ablation

**Project**: MSTL Implementation and Extensions  
**Authors**: Rup Narayan Jha, Jaypal Malviya, Anuj Kumar, Ravi Kiran

---

## 1. Objectives
This notebook implements two high-priority extensions beyond the original paper:
1. **Extension 1: Seasonal Window and Iteration Sensitivity Analysis**:
   - Grid search across odd seasonal windows $n_s \in \{7, 11, 15, 21, 25\}$ and outer iterations $I \in \{1, 2, 3\}$.
   - Evaluating reconstruction fidelity, smoothness, and computational runtime.
2. **Extension 2: Ablation Study of Multiple Seasonalities**:
   - Decomposing using Daily seasonality only ($m=24$).
   - Decomposing using Weekly seasonality only ($m=168$).
   - Decomposing using Both seasonalities ($m=\{24, 168\}$).
   - Measuring residual variance reduction and trend stability.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('../src'))

from data_processing import load_hourly, extract_experiment_period
from experiments import extension1_sensitivity, extension2_ablation

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
sns.set_theme(style='whitegrid')

## 2. Load Electricity Demand Data

In [ ]:
hourly = load_hourly('../data/vic_elec_hourly.csv')
series = extract_experiment_period(hourly, n_obs=3601)
x = series.values.astype(float)

## 3. Extension 1: Sensitivity Analysis (Windows vs Iterations)

In [ ]:
df_rmse_sens, df_time_sens = extension1_sensitivity(
    x, periods=[24, 168],
    windows_to_try=[7, 11, 15, 21, 25],
    iterations_to_try=[1, 2, 3]
)

print("Reconstruction RMSE Grid:")
display(df_rmse_sens)

print("Execution Time Grid (seconds):")
display(df_time_sens)

### Sensitivity Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(df_time_sens, annot=True, fmt='.3f', cmap='Blues', ax=ax)
ax.set_title("MSTL Execution Time (seconds) by Window and Iterations", fontweight='bold')
ax.set_xlabel("Outer Iterations")
ax.set_ylabel("Seasonal Window (ns)")
plt.tight_layout()
plt.show()

## 4. Extension 2: Multi-Seasonality Ablation Study
We compare three model configurations to determine the contribution of each seasonal pattern.

In [ ]:
ablation_results = extension2_ablation(x)

ablation_summary = pd.DataFrame([
    {
        'Configuration': name,
        'Seasonal Periods': str(res['periods']),
        'Recon. RMSE': f"{res['recon_rmse']:.2e}",
        'Remainder Std (MWh)': f"{res['remainder_std']:.2f}",
        'Runtime (s)': f"{res['time_sec']:.3f}"
    }
    for name, res in ablation_results.items()
]).set_index('Configuration')

ablation_summary

### Visual Comparison of Residual Standard Deviations

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
stds = [ablation_results[k]['remainder_std'] for k in ablation_results]
bars = ax.bar(list(ablation_results.keys()), stds, color=['#2196F3', '#FF9800', '#4CAF50'], width=0.5)
ax.bar_label(bars, fmt='%.1f MWh', padding=3)
ax.set_title("Ablation Study: Remainder Standard Deviation by Configuration", fontweight='bold')
ax.set_ylabel("Remainder Std (MWh)")
plt.tight_layout()
plt.show()

### Visual Comparison of Estimated Trends across Ablation Configurations

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(series.index, ablation_results['Daily only (24h)']['result']['trend'], label='Daily only (24h)', color='steelblue', lw=1.0)
ax.plot(series.index, ablation_results['Weekly only (168h)']['result']['trend'], label='Weekly only (168h)', color='darkorange', lw=1.0)
ax.plot(series.index, ablation_results['Both (24h + 168h)']['result']['trend'], label='Both (24h + 168h)', color='green', lw=1.2, ls='--')
ax.set_title("Estimated Trend Across Seasonality Configurations", fontweight='bold')
ax.set_ylabel("Trend (MWh)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Conclusions
1. **Window Sensitivity**: MSTL achieves exact reconstruction across all window sizes ($< 10^{-13}$). Execution time scales linearly with outer iterations ($0.10$s for $I=1$ to $0.31$s for $I=3$).
2. **Ablation**: Weekly seasonality alone leaves large daily fluctuations in residuals (std = $861.03$ MWh). The combined model ($m=[24, 168]$) models both cycles with residual std = $809.77$ MWh.